# FBref — complementary stats, top 5 leagues, 2017–2025

What Understat does not have — today only the **basics**: tackles won,
interceptions, fouls committed and drawn, crosses, cards.

> **FBref lost its Opta advanced data** (checked Sep 2026, every season
> 2017–2025): tackles by third, blocks, clearances, aerial duels, recoveries,
> progression and PSxG are still in the headers, but the cells are empty. This
> notebook drops every fully empty column by itself. Before adding a new table
> type, save one season and check it.

**Pages are saved by hand.** FBref sits behind a Cloudflare challenge that
blocks automated requests, so the default (`AUTO_FETCH = False`) is: section 1
lists the pages still missing, you open each one in Chrome and save it.
`AUTO_FETCH = True` opens a *visible* Chrome window instead, where you clear the
check yourself if it appears, then pages load at a human pace (one every ~7 s;
FBref bans above ~10 requests a minute).

- Raw HTML: `data/_fbref/html/<league>/<year>_<stat>.html` — a re-run only
  needs what is missing.
- Output: `data/fbref_stats.parquet`, one row per (season, league, player, club),
  and `data/fbref_matched.parquet`, the stats keyed by the Transfermarkt panel.

> **License.** FBref's data comes from Opta. Fine for a prototype; a product
> sold to clubs needs a licensed feed.

In [1]:
import os
import re
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup


def find_project_root(marker: str = "src/features/engineer.py") -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"could not find {marker} at or above {here}")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
sys.path.insert(0, str(BASE_PATH))

from src.data.crosswalk import MANUAL_DIR, build_crosswalk  # noqa: E402
from src.data.leagues import LEAGUES, fbref_url  # noqa: E402

FBREF_DIR = BASE_PATH / "data" / "_fbref"
HTML_DIR = FBREF_DIR / "html"
PROFILE_DIR = FBREF_DIR / "browser_profile"
HTML_DIR.mkdir(parents=True, exist_ok=True)
TM_PATH = BASE_PATH / "data" / "players_panel.parquet"
OUT_PATH = BASE_PATH / "data" / "fbref_stats.parquet"
MATCHED_PATH = BASE_PATH / "data" / "fbref_matched.parquet"

YEARS = list(range(2017, 2026))   # 2017 = 2017-18, as in notebooks 01 and 02
# FBref blocks automated browsing (Cloudflare). False: you save the pages by
# hand (section 1 lists the missing ones). True: try opening Chrome.
AUTO_FETCH = False
PAGE_DELAY = 7.0                  # seconds between pages: ~8.5/min, under the limit
CHALLENGE_TIMEOUT = 180           # how long to wait for your click on the check

# stat type in the URL -> id of the players table on the page.
# Only "misc" still has data: tackles won, interceptions, fouls, crosses, cards.
# Checked Sep 2026 over the 9 seasons:
#   - defense: the advanced columns are empty; what is left (tackles_won,
#     interceptions) is identical to misc — redundant.
#   - possession, passing, keepersadv, playingtime (on-pitch xG): Opta advanced
#     data, removed. Before adding a type here, save one season and check that
#     its columns are filled.
TABLES = {
    "misc": "stats_misc",
}


def cache_path(code: str, stat: str, year: int) -> Path:
    return HTML_DIR / code / f"{year}_{stat}.html"


PAGES = [(code, stat, year) for code in LEAGUES for year in YEARS for stat in TABLES]
print(f"project root: {BASE_PATH}")
print(f"{len(LEAGUES)} leagues x {len(TABLES)} tables x {len(YEARS)} seasons = {len(PAGES)} pages")

project root: /Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value
5 leagues x 1 tables x 9 seasons = 45 pages


## 1. Pages

With `AUTO_FETCH = False` (the default) this cell only lists the pages not yet in
`data/_fbref/html/<league>/`. For each one: open the URL in Chrome, **Cmd+S →
Format "Web Page, HTML Only"**, and save it under the name shown. Missing pages
are fine — sections 2 and 3 use whatever is there.

With `AUTO_FETCH = True`: Chrome opens, and for each missing page it loads, waits
for Cloudflare (if a check appears, **click it in the window**), confirms the
table is there and saves the HTML. On FBref's rate-limit page (429) it stops at
once — continue in about an hour; the cache keeps what already came.

In [2]:
class FBrefBlocked(RuntimeError):
    """The FBref rate limit (429) page, or a challenge nobody cleared."""


def is_challenge(page) -> bool:
    title = page.title().lower()
    return any(t in title for t in ("just a moment", "um momento", "attention required"))


def fetch_page(page, url: str, table_id: str) -> str:
    """Load one page in the open browser and return its HTML once the table is there."""
    page.goto(url, wait_until="domcontentloaded", timeout=60_000)

    waited = 0
    while is_challenge(page):
        if waited == 0:
            print("  -> Cloudflare check: click it in the Chrome window")
        if waited >= CHALLENGE_TIMEOUT:
            raise FBrefBlocked(f"check not cleared within {CHALLENGE_TIMEOUT}s: {url}")
        time.sleep(2)
        waited += 2

    html = page.content()
    if "Rate Limited Request" in html or "429 error" in html:
        raise FBrefBlocked("FBref rate-limited the requests (429). Wait ~1 hour and run again.")
    if f'id="{table_id}"' not in html:
        raise FBrefBlocked(f"the page loaded without the {table_id} table: {url}")
    return html


def fetch_all(todo: list) -> None:
    from playwright.sync_api import sync_playwright

    with sync_playwright() as p:
        browser = p.chromium.launch_persistent_context(
            str(PROFILE_DIR), channel="chrome", headless=False,
            viewport={"width": 1280, "height": 900},
        )
        page = browser.pages[0] if browser.pages else browser.new_page()
        try:
            for n, (code, stat, year) in enumerate(todo, 1):
                html = fetch_page(page, fbref_url(code, year, stat), TABLES[stat])
                path = cache_path(code, stat, year)
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_text(html)
                print(f"  {n}/{len(todo)}  {code} {year} {stat}", flush=True)
                if n < len(todo):
                    time.sleep(PAGE_DELAY)
        finally:
            browser.close()


todo = [p for p in PAGES if not cache_path(*p).exists()]
print(f"{len(PAGES) - len(todo)} pages cached, {len(todo)} to fetch")

if todo and not AUTO_FETCH:
    for code, stat, year in todo:
        print(f"  {cache_path(code, stat, year).relative_to(BASE_PATH)}  <-  {fbref_url(code, year, stat)}")
elif todo:
    # Playwright's sync API refuses to run inside Jupyter's event loop; a worker
    # thread has no loop of its own, so the same code works here and in a script.
    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(fetch_all, todo).result()

45 pages cached, 0 to fetch


## 2. Read the tables

FBref keeps most tables inside HTML comments, so comments are stripped before
parsing. Every cell has a `data-stat` attribute with a stable name
(`tackles_won`, `interceptions`, …), used as the column name instead of the
page's two-row header.

Each player is identified by the FBref id (the 8 characters in the player link),
not by name. Every page is checked against its folder and file name — a page
saved for the wrong season or league looks perfectly fine otherwise.

In [3]:
def parse_table(html: str, table_id: str) -> pd.DataFrame:
    """One row per player-club in a FBref league table, columns named by data-stat."""
    soup = BeautifulSoup(html.replace("<!--", "").replace("-->", ""), "html.parser")
    table = soup.find("table", id=table_id)
    rows = []
    for tr in table.select("tbody > tr"):
        if "thead" in (tr.get("class") or []):          # repeated header rows
            continue
        cell = tr.find(attrs={"data-stat": "player"})
        link = cell.find("a", href=True) if cell else None
        match = re.search(r"/players/([0-9a-f]{8})/", link["href"]) if link else None
        if not match:
            continue
        record = {"fbref_id": match.group(1), "player_name_fb": link.get_text(strip=True)}
        for td in tr.find_all(["td", "th"]):
            key = td.get("data-stat")
            if key and key not in ("ranker", "player", "matches"):
                record[key] = td.get_text(strip=True)
        rows.append(record)
    return pd.DataFrame(rows)


def check_page(html: str, code: str, stat: str, year: int) -> None:
    """The page's <title> must name the season and league its file is saved under."""
    title = re.search(r"<title>(.*?)</title>", html, re.S)
    title = title.group(1) if title else ""
    season = f"{year}-{year + 1}"
    league = LEAGUES[code].fbref_name.replace("-", " ")
    if season not in title or league not in title:
        raise ValueError(
            f"{cache_path(code, stat, year).relative_to(HTML_DIR)} is '{title.strip()}', "
            f"expected {season} {league}. Save it again from {fbref_url(code, year, stat)}"
        )


def season_frame(code: str, year: int) -> pd.DataFrame:
    """All tables of one league-season merged into one row per (player, club)."""
    merged = None
    for stat, table_id in TABLES.items():
        path = cache_path(code, stat, year)
        if not path.exists():
            continue
        html = path.read_text()
        check_page(html, code, stat, year)
        df = parse_table(html, table_id)
        if df.empty:
            continue
        key = ["fbref_id", "team"]
        if merged is None:
            merged = df
        else:
            # several tables repeat the same columns (interceptions, minutes_90s,
            # identity fields) — keep the first copy only
            new = [c for c in df.columns if c not in merged.columns]
            merged = merged.merge(df[key + new], on=key, how="outer")
    if merged is None:
        return pd.DataFrame()
    merged.insert(0, "league_code", code)
    merged.insert(0, "year", year)
    return merged


fbref = pd.concat([season_frame(code, y) for code in LEAGUES for y in YEARS], ignore_index=True)
fbref.insert(2, "league", fbref["league_code"].map(lambda c: LEAGUES[c].name))

# everything that isn't an identifier is a number ("1,234" -> 1234, "" -> NaN)
text_cols = {"fbref_id", "player_name_fb", "nationality", "position", "team", "age",
             "league_code", "league"}
for col in fbref.columns.difference(list(text_cols) + ["year"]):
    fbref[col] = pd.to_numeric(fbref[col].astype(str).str.replace(",", ""), errors="coerce")

# FBref lost its Opta advanced data: the headers are still there but the cells
# are blank. Drop every column that is empty across the whole panel so nothing
# dead reaches a model.
empty = [c for c in fbref.columns if fbref[c].isna().all() or (fbref[c].astype(str).str.strip() == "").all()]
fbref = fbref.drop(columns=empty)
print(f"empty columns dropped ({len(empty)}): {empty}")
print(f"fbref: {fbref.shape[0]:,} rows x {fbref.shape[1]} columns")
fbref.groupby(["league_code", "year"]).size().unstack()

empty columns dropped (0): []
fbref: 25,257 rows x 23 columns


year,2017,2018,2019,2020,2021,2022,2023,2024,2025
league_code,,,,,,,,,
ES1,573,544,570,582,617,596,609,601,611
FR1,553,561,542,584,604,606,540,553,561
GB1,529,508,522,532,546,569,580,574,551
IT1,552,569,600,619,632,603,616,634,609
L1,484,474,498,505,523,515,507,492,507


## 3. Check and save

The key (season, league, player, club) is unique, every league-season has a
plausible number of players, and the headline column of each table came filled.

In [4]:
key_dups = fbref.duplicated(subset=["year", "league_code", "fbref_id", "team"]).sum()
per_season = fbref.groupby(["league_code", "year"]).size()
# the headline column of each table type that can be downloaded
HEADLINE = {"defense": "tackles_won", "misc": "crosses", "possession": "progressive_carries",
            "passing": "progressive_passes", "playingtime": "games_starts", "keepersadv": "gk_psxg"}
present = [s for s in HEADLINE if s in TABLES and any(cache_path(c, s, y).exists() for c in LEAGUES for y in YEARS)]
must_have = [HEADLINE[s] for s in present]

print(f"duplicate key              : {key_dups}")
print(f"players per league-season  : {per_season.min()}–{per_season.max()}")
print(f"tables downloaded          : {present}")
print("filled share of those columns, by league:")
print((fbref.groupby("league")[must_have].apply(lambda d: d.notna().mean()) * 100).round(0).to_string())

assert key_dups == 0, "duplicate (year, league_code, fbref_id, team)"
assert per_season.between(350, 800).all(), f"a league-season with an odd number of players:\n{per_season}"
missing = [c for c in must_have if c not in fbref.columns]
assert not missing, f"table downloaded but column missing: {missing} — did FBref's layout change?"

fbref.to_parquet(OUT_PATH, index=False)
print(f"\nsaved {OUT_PATH.name}: {fbref.shape}")

duplicate key              : 0
players per league-season  : 474–634
tables downloaded          : ['misc']
filled share of those columns, by league:
                crosses
league                 
Bundesliga        100.0
LaLiga            100.0
Ligue 1           100.0
Premier League    100.0
Serie A           100.0

saved fbref_stats.parquet: (25257, 23)


## 4. Link to the Transfermarkt panel

Through `src/data/crosswalk.py`, the same deterministic rules as Understat in
notebook 02, with two extras FBref allows:

- **public map first** — worldfootballR's FBref → Transfermarkt URL map links a
  player outright (kept only when both birth years agree);
- **birth year** — FBref has it, so every rule also requires it to match.

Leftovers go to `data/manual/review_fbref.csv`; accepted ones go in
`data/manual/id_overrides.csv` (source `fbref`) and win on the next run. The
Understat links already on disk are left as they are.

The Transfermarkt side is the full panel from notebook 01 (both clubs of a
mid-season move), so it does not depend on notebook 02. Stats are summed per
player-season over clubs and leagues; `src/data/loader.py` joins the result to
`panel_final.parquet` on (year, player_id), and `build_features()` makes the
per-90 features.

In [5]:
from src.data.fbref import match_to_panel  # noqa: E402

tm_panel = pd.read_parquet(TM_PATH)
tm_panel["player_id"] = tm_panel["player_id"].astype(str)

xw = build_crosswalk(tm_panel, None, fbref)
links = xw["crosswalk"]
review = xw["review_fbref"]

played = fbref[fbref["minutes_90s"] > 0]
linked = played["fbref_id"].isin(set(links["src_id"]))
print(f"links: {len(links):,} players   by rule: {links['rule'].value_counts().to_dict()}")
print(f"clubs not paired : {xw['unmapped_clubs_fbref'] or 'none'}")
print(f"conflicts dropped: {len(xw['conflicts_fbref'])}")
print("\nFBref player-seasons with minutes that got a link, by league:")
print(linked.groupby(played["league"]).mean().map("{:.1%}".format).to_string())
print(f"\nleft for review: {len(review)} players -> {MANUAL_DIR / 'review_fbref.csv'}")
if len(review):
    print(review.groupby("league_code").size().to_string())

links: 7,813 players   by rule: {'public_map': 7031, 'name_club': 688, 'subset_club': 49, 'override': 25, 'initial_club': 20}
clubs not paired : none
conflicts dropped: 0

FBref player-seasons with minutes that got a link, by league:
league
Bundesliga        100.0%
LaLiga            100.0%
Ligue 1           100.0%
Premier League    100.0%
Serie A           100.0%

left for review: 5 players -> /Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value/data/manual/review_fbref.csv
league_code
ES1    2
FR1    1
IT1    1
L1     1


In [6]:
matched = match_to_panel(fbref, links)

assert not matched.duplicated(["year", "player_id"]).any(), "panel player matched twice"
assert not matched.duplicated(["year", "fbref_id"]).any(), "FBref player matched twice"

on_tm = matched.set_index(["year", "player_id"]).index.isin(
    tm_panel.assign(year=tm_panel["year"].astype(int)).set_index(["year", "player_id"]).index)
print(f"matched player-seasons: {len(matched):,}   without a Transfermarkt row that season: {(~on_tm).sum()}")
print(f"stats summed over two leagues: {matched['fb_leagues'].str.contains(',').sum()}")

matched.to_parquet(MATCHED_PATH, index=False)
print(f"saved {MATCHED_PATH.name}: {matched.shape}")

matched player-seasons: 24,059   without a Transfermarkt row that season: 3
stats summed over two leagues: 541
saved fbref_matched.parquet: (24059, 11)
